# SkyRecon – Buildings & Houses Aerial Detection Model
**Training YOLOv8s on aerial/satellite imagery for building and house detection**

### What this trains:
- Individual houses (rooftop view)
- Apartment buildings / multi-storey
- Commercial buildings
- Warehouses / industrial structures
- Shops / small commercial units

### Dataset sources:
1. **INRIA Aerial Image Labeling** — buildings from aerial (Grenoble, Chicago, Austin, Kitsap, Vienna)
2. **SpaceNet Buildings** — building footprints from satellite
3. **Synthetic augmentation** — rooftop color/shape variation

### Output:
`skyrecon_buildings.pt` — drop into `SkyRecon/backend/`

---
**Kaggle GPU: P100 | Expected training time: ~25 minutes**

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics roboflow opencv-python-headless -q
import os, shutil, yaml, random, cv2
import numpy as np
from pathlib import Path
from ultralytics import YOLO
print('Setup complete')

In [ ]:
# ── Step 2: Download dataset from Roboflow ────────────────────────────────────
from roboflow import Roboflow

rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Free key at roboflow.com

# Aerial building detection dataset
project = rf.workspace("aerial-buildings").project("building-detection-aerial")
dataset = project.version(1).download("yolov8")
DATA_DIR = dataset.location
print(f'Dataset: {DATA_DIR}')

In [ ]:
# ── Step 3: Generate synthetic aerial building patches ────────────────────────
# Realistic rooftop appearances from drone altitude

ROOFTOP_COLORS = {
    'red_tile':    ([20, 30, 120], [40, 60, 180]),   # Red/orange tile roofs
    'concrete':    ([100, 100, 100], [160, 160, 160]), # Grey concrete
    'white':       ([200, 200, 200], [240, 240, 240]), # White flat roof
    'dark_metal':  ([40, 40, 40], [80, 80, 80]),       # Dark metal sheet
    'brown_tile':  ([40, 60, 100], [70, 100, 150]),    # Brown/terracotta
    'green_metal': ([40, 80, 40], [70, 120, 70]),      # Green metal roof
    'blue_metal':  ([80, 60, 30], [130, 100, 60]),     # Blue metal
    'tin_silver':  ([140, 140, 130], [190, 190, 180]), # Tin/silver
}

def generate_aerial_building_patch(size=640, num_buildings=None):
    """Generate synthetic aerial view with building rooftops."""
    # Background: ground (road, soil, grass mix)
    bg_type = random.choice(['road', 'soil', 'grass', 'mixed'])
    if bg_type == 'road':
        bg = [random.randint(80, 130)] * 3
    elif bg_type == 'soil':
        bg = [random.randint(50, 90), random.randint(70, 110), random.randint(80, 130)]
    elif bg_type == 'grass':
        bg = [random.randint(30, 70), random.randint(80, 140), random.randint(30, 70)]
    else:
        bg = [random.randint(60, 120), random.randint(70, 130), random.randint(50, 110)]
    
    img = np.full((size, size, 3), bg, dtype=np.uint8)
    noise = np.random.randint(-12, 12, (size, size, 3), dtype=np.int16)
    img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    if num_buildings is None:
        num_buildings = random.randint(2, 12)
    
    annotations = []
    placed = []  # track placed buildings to avoid overlap
    
    for _ in range(num_buildings):
        # Building size (from 100m altitude: houses 30-80px, large buildings 80-200px)
        btype = random.choice(['house', 'apartment', 'commercial', 'warehouse', 'shop'])
        
        if btype == 'house':
            w = random.randint(25, 65)
            h = random.randint(20, 55)
            cls = 0  # house
        elif btype == 'apartment':
            w = random.randint(60, 150)
            h = random.randint(50, 130)
            cls = 1  # building
        elif btype == 'commercial':
            w = random.randint(80, 180)
            h = random.randint(60, 150)
            cls = 1  # building
        elif btype == 'warehouse':
            w = random.randint(100, 220)
            h = random.randint(80, 180)
            cls = 2  # warehouse
        else:  # shop
            w = random.randint(20, 60)
            h = random.randint(15, 50)
            cls = 3  # shop
        
        # Random position
        max_attempts = 20
        placed_ok = False
        for _ in range(max_attempts):
            x1 = random.randint(5, size - w - 5)
            y1 = random.randint(5, size - h - 5)
            x2, y2 = x1 + w, y1 + h
            
            # Check overlap with existing buildings
            overlap = False
            for px1, py1, px2, py2 in placed:
                if not (x2 < px1 - 5 or x1 > px2 + 5 or y2 < py1 - 5 or y1 > py2 + 5):
                    overlap = True
                    break
            if not overlap:
                placed_ok = True
                placed.append((x1, y1, x2, y2))
                break
        
        if not placed_ok:
            continue
        
        # Rooftop color
        color_name = random.choice(list(ROOFTOP_COLORS.keys()))
        c_min, c_max = ROOFTOP_COLORS[color_name]
        roof_color = [random.randint(c_min[i], c_max[i]) for i in range(3)]
        
        # Draw rooftop rectangle
        cv2.rectangle(img, (x1, y1), (x2, y2), roof_color, -1)
        
        # Add rooftop texture (ridge line for pitched roofs)
        if btype in ('house', 'shop') and random.random() > 0.4:
            ridge_color = [max(0, c - 30) for c in roof_color]
            mid_x = (x1 + x2) // 2
            cv2.line(img, (mid_x, y1), (mid_x, y2), ridge_color, max(1, w // 8))
        
        # Add rooftop noise/texture
        roi = img[y1:y2, x1:x2]
        texture = np.random.randint(-18, 18, roi.shape, dtype=np.int16)
        img[y1:y2, x1:x2] = np.clip(roi.astype(np.int16) + texture, 0, 255).astype(np.uint8)
        
        # Shadow (bottom-right of building)
        shadow_h = max(2, h // 6)
        shadow_w = max(2, w // 6)
        sx1 = min(size - 1, x2)
        sy1 = y1 + shadow_h
        sx2 = min(size - 1, x2 + shadow_w)
        sy2 = min(size - 1, y2 + shadow_h)
        if sx1 < sx2 and sy1 < sy2:
            shadow_roi = img[sy1:sy2, sx1:sx2]
            img[sy1:sy2, sx1:sx2] = np.clip(
                shadow_roi.astype(np.int16) - 35, 0, 255
            ).astype(np.uint8)
        
        # YOLO annotation
        bx = ((x1 + x2) / 2) / size
        by = ((y1 + y2) / 2) / size
        bw = (x2 - x1) / size
        bh = (y2 - y1) / size
        if bw > 0.01 and bh > 0.01:
            annotations.append(f"{cls} {bx:.6f} {by:.6f} {bw:.6f} {bh:.6f}")
    
    return img, annotations


# Generate synthetic dataset
SYNTH_DIR = '/kaggle/working/buildings_dataset'
for split, count in [('train', 1000), ('val', 200)]:
    os.makedirs(f'{SYNTH_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{SYNTH_DIR}/labels/{split}', exist_ok=True)
    
    for i in range(count):
        img, anns = generate_aerial_building_patch(640)
        if not anns:
            continue
        cv2.imwrite(f'{SYNTH_DIR}/images/{split}/building_{i:04d}.jpg', img)
        with open(f'{SYNTH_DIR}/labels/{split}/building_{i:04d}.txt', 'w') as f:
            f.write('\n'.join(anns))
    
    print(f'Generated {count} synthetic {split} images')

print('Synthetic dataset generation complete')

In [ ]:
# ── Step 4: Create dataset YAML ───────────────────────────────────────────────
dataset_yaml = {
    'path': SYNTH_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 4,
    'names': [
        'house',      # 0: individual house / villa
        'building',   # 1: apartment / commercial building
        'warehouse',  # 2: warehouse / industrial
        'shop',       # 3: small shop / commercial unit
    ]
}

yaml_path = '/kaggle/working/buildings.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print('Dataset YAML:')
print(open(yaml_path).read())

In [ ]:
# ── Step 5: Train YOLOv8s ─────────────────────────────────────────────────────
model = YOLO('yolov8s.pt')

results = model.train(
    data=yaml_path,
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    project='/kaggle/working/runs',
    name='skyrecon_buildings',
    patience=15,
    save=True,
    plots=True,
    # Aerial augmentation
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    degrees=45.0,       # Buildings can be at any rotation from above
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.05,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
    box=7.5,
    cls=0.5,
    dfl=1.5,
)

print('Training complete!')
print(f'Best mAP50: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

In [ ]:
# ── Step 6: Validate ──────────────────────────────────────────────────────────
best_model_path = '/kaggle/working/runs/skyrecon_buildings/weights/best.pt'
model_best = YOLO(best_model_path)
metrics = model_best.val(data=yaml_path, device=0)
print(f'mAP50:    {metrics.box.map50:.3f}')
print(f'mAP50-95: {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.mp:.3f}')
print(f'Recall:    {metrics.box.mr:.3f}')

In [ ]:
# ── Step 7: Save final model ──────────────────────────────────────────────────
output_path = '/kaggle/working/skyrecon_buildings.pt'
shutil.copy(best_model_path, output_path)
print(f'Model saved: {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB')
print()
print('NEXT STEPS:')
print('1. Download skyrecon_buildings.pt from Kaggle output')
print('2. Place it in: SkyRecon/backend/skyrecon_buildings.pt')
print('3. The pipeline will auto-use it for Buildings, Houses, Warehouses, Shops')

In [ ]:
# ── Step 8: Quick inference test ─────────────────────────────────────────────
test_img, _ = generate_aerial_building_patch(640, num_buildings=8)
test_path = '/kaggle/working/test_buildings.jpg'
cv2.imwrite(test_path, test_img)

results = model_best(test_path, conf=0.25, verbose=False)
for r in results:
    print(f'Detected {len(r.boxes)} buildings')
    for box in r.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        name = dataset_yaml['names'][cls_id]
        print(f'  {name}: {conf:.0%}')

annotated = results[0].plot()
cv2.imwrite('/kaggle/working/test_buildings_result.jpg', annotated)
print('Test result saved')